# 02. 가설검정 기초

- **분석 배경**
  - 기술통계 실습에서 점포 유형에 따라 월평균 매출이 달라 보였지만, 표본에서 본 차이가 우연인지 여부를 기술통계만으로 판단할 수 없습니다.
  - 점포 매출과 신규 고객 재구매율에서 보이는 차이가 모집단에서도 존재한다고 말할 수 있는지 가설검정으로 확인하고자 합니다.
- **분석 대상**
  - 점포: 100개 점포의 2025년 점포별 월평균 매출입니다.
  - 고객: 2025년 7 ~ 12월에 첫 구매를 한 신규 고객 중 6차월 말까지 탈퇴하지 않은 고객입니다.
- **분석 질문**
  - 소형 점포와 중형 점포의 월평균 매출은 다른가?
  - 소형·중형·대형 점포의 월평균 매출은 다른가? 지역에 따라서는 어떤가?
  - 주 이용 채널이나 연령대에 따라 재구매율이 다른가?
- **사용 데이터**
  - `store_month_clean.pkl`(기술통계 실습에서 결합·보정한 점포 × 월 2,400행)
  - `repurchase.csv`(`new_customers.csv`와 `transactions.csv`를 결합·집계한 신규 고객별 구매 행동 9,173명)
    - 만드는 과정은 `appendix_a_rfm_features.ipynb` 파일을 참조하세요.
- **분석 단위**: 점포(두 집단 비교, 분산분석), 신규 고객(교차분석)

**학습 목표**

1. 정규성과 등분산성을 확인하고, 등분산 가정이 필요한지에 따라 Student t-검정과 Welch t-검정을 구분할 수 있다.
2. 독립표본 t-검정, Welch t-검정, Mann-Whitney U 검정 결과를 해석할 수 있다.
3. 분산분석과 사후분석으로 세 집단 이상의 평균을 비교할 수 있다.
4. 교차분석(카이제곱 검정)으로 두 범주형 변수의 관련성을 확인할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
from scipy import stats
import hds

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 데이터 불러오기

기술통계 실습에서 결합·보정해 저장한 pkl 파일을 불러옵니다.<br>
pkl 파일이 없으면 기술통계 실습 노트북을 먼저 실행합니다.

In [ ]:
# 기술통계 실습에서 저장한 pkl 파일을 읽고 df에 할당합니다.
df = pd.read_pickle(filepath_or_buffer='../data/store_month_clean.pkl')

# df의 처음 5행과 마지막 5행을 확인합니다.
df

In [ ]:
# df의 정보를 확인합니다.
df.info()

신규 고객의 재구매 분석용 csv 파일을 불러옵니다.

In [ ]:
# 재구매 분석용 csv 파일을 읽고 repurchase에 할당합니다.
repurchase = pd.read_csv(filepath_or_buffer='../data/repurchase.csv', encoding='utf-8-sig')

# repurchase의 처음 5행과 마지막 5행을 확인합니다.
repurchase

In [ ]:
# repurchase의 정보를 확인합니다.
repurchase.info()

## 1. 가설검정 절차

이 실습의 모든 검정은 다음 순서로 진행합니다.

1. 귀무가설과 대립가설을 세웁니다.
2. 유의수준을 정합니다. 이번 실습에서는 0.05로 설정합니다.
3. 데이터가 검정 방법의 가정(정규성, 등분산성 등)을 만족하는지 확인하고 검정 방법을 선택합니다.
4. 검정통계량과 유의확률을 계산합니다.
5. 유의확률이 유의수준보다 작으면 귀무가설을 기각하고, 결과를 업무 언어로 해석합니다.

가설검정의 판단에는 두 가지 오류가 있고, 유의수준과 검정력은 이 오류와 연결됩니다.

| 용어 | 뜻 |
|---|---|
| 제1종 오류 (α) | 귀무가설이 참인데 기각하는 오류입니다. 유의수준은 이 확률의 상한입니다. |
| 제2종 오류 (β) | 귀무가설이 거짓인데 기각하지 못하는 오류입니다. |
| 검정력 (1 − β) | 귀무가설이 거짓일 때 기각할 확률입니다. 보통 0.8 이상이 되도록 실험을 설계합니다. |

집단을 비교하는 검정은 다음과 같이 나뉩니다. 앞의 두 열은 **평균**을, 마지막 열은 **순위**를 비교합니다.

| 구분 | 등분산 가정이 필요한 방법 | 등분산 가정이 필요 없는 방법 | 순위로 비교하는 방법 |
|---|---|---|---|
| 두 집단 | Student t-검정 | Welch t-검정 | Mann-Whitney U 검정 |
| 세 집단 이상 | 일반 분산분석 | Welch 분산분석 | Kruskal-Wallis 검정 |
| 사후분석 | Tukey's HSD | Games-Howell 검정 | Dunn 검정 |

- Welch 방법은 분산이 같을 때도 손해가 거의 없으므로 **기본으로 사용합니다.**
  - 정규성 검정과 등분산성 검정 결과는 데이터를 이해하는 참고 정보이며, 검정 방법을 자동으로 결정하지 않습니다.
- 정규성을 만족하지 않아도 표본이 크면 표본평균의 분포가 정규분포에 가까워지므로 t-검정을 쓸 수 있습니다.
- 순위로 비교하는 방법은 평균이 아니라 '한 집단의 값이 다른 집단보다 큰 경향이 있는가'(순위·분포의 차이)를 검정합니다.
  - 평균 검정과 질문이 다르므로 평균 검정의 결론을 대신하지 않습니다. 평균이 관심 지표이면 평균 검정으로 판단하고, 순위 검정은 분포의 차이를 보는 보조 정보로 씁니다.

## 2. 두 집단 비교: t-검정과 Mann-Whitney U 검정

기술통계 실습에서 점포 유형별 월평균 매출과 신뢰구간을 살펴봤습니다.<br>
먼저 소형 점포와 중형 점포 두 유형의 월평균 매출이 다른지 검정합니다.

**질문**: 점포별 2025년 월평균 매출 테이블을 만듭니다.

In [ ]:
# df에서 2025년 행만 선택하고 store_2025에 할당합니다.
store_2025 = df.loc[df['ym'].str.startswith('2025'), :]

# store_2025에서 점포별 월평균 매출을 계산하고 store_2025에 재할당합니다.
# [참고] 점포 유형과 지역은 점포마다 하나로 정해진 값이므로 묶음 기준에 넣어도 점포별로 묶이는 것은 같고 결과에 열로 남습니다.
store_2025 = store_2025.groupby(by=['store_id', 'store_type', 'region'], as_index=False)['sales'].mean()

# store_2025의 행 개수와 열 개수를 확인합니다.
store_2025.shape

In [ ]:
# store_2025의 처음 5행을 확인합니다.
store_2025.head()

**해석**
- 점포 100개, 한 점포가 한 행인 테이블입니다.
- 독립표본 t-검정과 분산분석 모두 관측값이 서로 독립이어야 합니다.
  - 같은 점포가 반복되는 `점포 × 월`이 아니라 `점포` 단위로 비교합니다.

**질문**: 소형 점포와 중형 점포의 월평균 매출은 어떻게 다른가?

In [ ]:
# 점포 유형의 고유값을 확인합니다.
store_2025['store_type'].unique()

In [ ]:
# 점포 유형을 서열형으로 변환합니다.
# [참고] 서열형 변수는 고유값별로 순위를 유지하므로 오름차순 또는 내림차순 정렬한 결과를 통제하기 쉽습니다.
store_2025['store_type'] = pd.Categorical(values=store_2025['store_type'], categories=['소형', '중형', '대형'], ordered=True)

# store_2025의 열별 자료형을 확인합니다.
store_2025.dtypes

In [ ]:
# 소형 점포와 중형 점포만 선택하고 store_sm에 할당합니다.
store_sm = store_2025.loc[store_2025['store_type'].isin(values=['소형', '중형']), :]

# 점포 유형별 매출의 기술통계량을 확인합니다.
store_sm.groupby(by='store_type')['sales'].describe().round(decimals=1)

In [ ]:
# 1행 1열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4), dpi=100)

# 점포 유형별 매출 분포를 커널 밀도 추정 곡선으로 그립니다.
# [참고] common_norm=False는 집단마다 곡선 아래 면적이 1이 되도록 하여 점포 수가 달라도 모양을 비교할 수 있게 합니다.
sns.kdeplot(data=store_sm, x='sales', hue='store_type', hue_order=['소형', '중형'], fill=True, common_norm=False, ax=ax)
ax.set(title='소형·중형 점포의 월평균 매출 (2025년)', xlabel='월평균 매출 (백만 원)', ylabel='밀도')
plt.show()

**해석**
- 두 유형은 모두 평균과 중위수가 비슷하고, 좌우로 크게 치우치지 않은 분포를 보입니다.
  - 소형 점포보다 중형 점포의 평균이 더 큽니다.
  - 중형 점포의 매출이 소형 점포보다 전체적으로 오른쪽에 있습니다.
- 중형 점포의 곡선은 봉우리가 두 개처럼 보입니다.
  - 현실에서 점포를 나누는 기준은 명확하지 않을 수 있습니다.
  - 예를 들어 한 집단 안에서 성격이 다른 점포 무리(수도권과 비수도권 등)가 섞여 있으면 봉우리가 여러 개 나타날 수 있습니다.
  - 점포 수가 적으면 곡선이 작은 뭉침에도 쉽게 출렁이므로 모양만으로 판단하지 말고 정규성 검정 결과와 함께 봅니다.
- 두 유형의 차이가 우연으로 생길 수 있는 수준인지 검정합니다.

**질문**: 두 유형의 매출은 정규분포를 따르는가?

In [ ]:
# 점포 유형별 매출의 정규성 검정을 실행합니다.
# [참고] 기본값은 Shapiro-Wilk 검정(method='shapiro')입니다.
pg.normality(data=store_sm, dv='sales', group='store_type')

**해석**
- 귀무가설은 '매출이 정규분포를 따른다'입니다.
  - 두 유형 모두 `normal`이 True(유의확률 ≥ 0.05)이므로 정규분포를 따르지 않는다고 볼 근거가 없습니다.
- 점포 수가 30개, 40개로 많지 않으므로 정규성 확인이 중요합니다.
  - 표본이 작으면 표본평균의 분포가 정규분포에 가까워진다고 기대하기 어렵기 때문입니다.

**질문**: 두 유형의 매출은 분산이 같은가?

In [ ]:
# 점포 유형별 매출의 등분산성 검정을 실행합니다.
# [참고] Levene 검정은 기본으로 각 값과 집단 중위수의 거리로 분산을 비교합니다.
pg.homoscedasticity(data=store_sm, dv='sales', group='store_type', method='levene')

**해석**
- 귀무가설은 '두 유형의 분산이 같다'입니다.
  - `equal_var`가 True(유의확률 ≥ 0.05)이므로 두 유형의 분산이 다르다고 볼 근거가 없습니다.
- 등분산 가정이 필요한 Student t-검정을 써도 되는 상황이지만, Welch t-검정을 기본으로 판단하고 Student t-검정 결과와 비교합니다.

**질문**: 소형 점포와 중형 점포의 평균 매출은 차이가 있는가?
- 귀무가설: 두 유형의 평균 매출은 같다.
- 대립가설: 두 유형의 평균 매출은 같지 않다.

In [ ]:
# 중형 점포의 매출을 y1에 할당합니다.
y1 = store_sm.loc[store_sm['store_type'].eq(other='중형'), 'sales']

# 소형 점포의 매출을 y2에 할당합니다.
y2 = store_sm.loc[store_sm['store_type'].eq(other='소형'), 'sales']

# 등분산을 가정하지 않는 t-검정(Welch)을 실행합니다.
# [참고] correction=True는 등분산을 가정하지 않는 독립표본 t-검정을 실행합니다.
pg.ttest(x=y1, y=y2, correction=True)

In [ ]:
# 두 유형의 평균 매출과 그 차이를 출력합니다.
print(f'평균 매출: 중형 = {y1.mean():,.1f}, 소형 = {y2.mean():,.1f}, 차이 = {y1.mean() - y2.mean():,.1f} (백만 원)')

**해석**
- `p_val`(유의확률)이 0.05보다 작으므로 귀무가설을 기각합니다.
  - 양측 검정이므로 중형 점포의 평균 매출이 소형 점포와 같지 않다고 판단합니다.
  - t-통계량이 양수이므로 중형 점포의 평균 매출이 소형 점포보다 큰 경우입니다.
- `CI95`는 평균 차이(중형 − 소형)의 95% 신뢰구간입니다.
  - 0을 포함하지 않으므로 두 집단의 평균 차이가 0이 아니라는 것을 의미합니다.
- `cohen_d`(효과 크기)도 매우 크게 나왔습니다.
  - 0.8 이상이면 통계적으로뿐만 아니라 실무적으로도 큰 차이입니다.
- `power`(검정력)는 이번에 관측한 효과 크기가 실제 차이라고 가정하고 계산한 **사후 검정력**입니다.
  - 1.0은 반올림한 값입니다.
  - 두 집단의 평균 차이가 실제로 이만큼이라면 같은 점포 수로 같은 검정을 반복해도 거의 매번 귀무가설을 기각한다는 뜻입니다.
- 검정 결과 표의 `power`는 사후 검정력이므로 참고만 합니다.
  - 사전 검정력은 쿠폰 A/B 테스트 실습에서 실험 전에 표본 크기를 정할 때 사용합니다.

- 검정력은 효과 크기를 어디에서 가져오느냐에 따라 의미가 다릅니다.

  | 구분 | 효과 크기 | 용도 |
  |---|---|---|
  | **사전 검정력** | 실험 전에 '이 정도 차이는 잡아내고 싶다'는 목표 효과 크기 | 실험 전 표본 크기를 정할 때 사용 |
  | **사후 검정력** | 실험 후에 관측한 효과 크기 | 유의확률을 다른 눈금으로 다시 읽은 값이라 새로운 정보가 없음 |

  [참고] 관측한 효과 크기는 검정통계량에서 나온 값이므로 사후 검정력은 유의확률이 0.05이면 약 0.5이고 유의확률이 작을수록 자동으로 커집니다.

**질문**: 등분산 가정이 필요한 Student t-검정으로도 같은 결론이 나오는가?

In [ ]:
# 등분산을 가정한 t-검정(Student)을 실행합니다.
# [참고] correction=False는 등분산을 가정한 독립표본 t-검정을 실행합니다.
# 기본값 correction='auto'는 두 집단의 표본 크기가 다를 때 Welch t-검정을 실행합니다.
pg.ttest(x=y1, y=y2, correction=False)

**해석**
- 두 유형의 점포 수가 다르므로 Welch t-검정과 t-통계량, `dof`(자유도)가 조금 다르지만 결론은 같습니다.
  - 등분산성 검정에서 분산이 다르다고 볼 근거가 없었으므로 두 방법의 결과가 거의 같은 것이 자연스럽습니다.
- Welch t-검정은 분산이 같을 때도 손해가 거의 없고 분산이 다를 때는 더 정확하므로 **기본으로 사용합니다.**

**질문**: 순위로 비교하는 Mann-Whitney U 검정에서도 중형 점포의 매출이 큰 경향이 있는가?

In [ ]:
# 순위로 비교하는 Mann-Whitney U 검정을 실행합니다.
pg.mwu(x=y1, y=y2)

**해석**
- Mann-Whitney U 검정은 값 자체 대신 **순위**를 비교하므로 극단값의 영향을 덜 받습니다.
  - 평균 차이가 아니라 '한 집단의 값이 다른 집단보다 큰 경향이 있는가'를 검정합니다.
- 현재 데이터는 정규성을 만족하므로 평균 차이는 독립표본 t-검정으로 판단합니다.
  - Mann-Whitney U 검정도 `p_val`(유의확률)이 0.05보다 작아, 순위로 봐도 중형 점포의 매출이 큰 경향이 뚜렷합니다.
- `CLES`는 중형 점포와 소형 점포를 하나씩 무작위로 뽑았을 때 중형 점포의 매출이 더 클 확률입니다.
  - 1에 가까울수록 두 유형이 뚜렷하게 구분됩니다.

## 3. 세 집단 이상 비교: 분산분석(ANOVA)과 사후분석

세 개 이상의 집단 평균을 비교할 때는 분산분석을 씁니다.<br>
앞에서 비교한 소형·중형에 대형을 더해 세 유형의 매출을 비교하고, 지역별 매출도 비교합니다.

**질문**: 점포 유형(대형·중형·소형)별 평균 매출은 어떻게 다른가?

In [ ]:
# 점포 유형별 매출의 도수, 평균 및 표준편차를 확인합니다.
pd.pivot_table(data=store_2025, values='sales', index='store_type', aggfunc=['count', 'mean', 'std']) \
    .round(decimals=1)

In [ ]:
# 점포 유형별 매출 분포를 상자 그림으로 시각화합니다.
plt.figure(figsize=(6, 4), dpi=100)
sns.boxplot(data=store_2025, x='store_type', y='sales', boxprops={'facecolor': 'silver'}, medianprops={'color': 'red'})
plt.title(label='점포 유형별 월평균 매출 (2025년)')
plt.xlabel(xlabel='점포 유형')
plt.ylabel(ylabel='월평균 매출 (백만 원)')
plt.show()

**해석**
- 점포 유형은 매장 면적을 기준으로 나눈 것입니다.
  - 대형 점포의 매출이 높게 나오는 것은 자연스럽습니다.
- 상자 안 빨간 선은 각 집단의 중위수입니다.
  - 소형 약 2,300백만 원, 중형 약 3,300백만 원, 대형 약 4,900백만 원으로 규모가 커질수록 뚜렷하게 높아집니다.
- 세 점포 유형의 상자가 서로 겹치지 않습니다.
  - 점포 유형 안의 차이보다 점포 유형 간 차이가 훨씬 크다는 것을 의미합니다.
- 점포의 규모가 커질수록 상자와 수염이 길어집니다.
  - 피벗 테이블에서도 대형 점포의 표준편차가 소형 점포의 두 배를 넘으므로 등분산성을 만족하지 않을 수 있습니다.
- 대형 점포는 위쪽에 이상값(극단값)이 두 곳 있고, 중위수가 상자 아래쪽에 있어 평균이 중위수보다 큽니다.
  - 분포가 오른쪽으로 치우쳐 있으므로 정규성도 따로 확인해야 합니다.
- 상자 그림만으로도 점포 유형별 평균 차이가 커 보입니다.
  - 정규성과 등분산성을 확인한 뒤 세 집단의 평균 차이가 통계적으로 유의한지 검정합니다.

**질문**: 분산분석 전에 점포 유형별 매출의 정규성과 등분산성을 확인합니다.

In [ ]:
# 점포 유형별 매출의 정규성 검정을 실행합니다.
pg.normality(data=store_2025, dv='sales', group='store_type')

In [ ]:
# 점포 유형별 매출의 등분산성 검정을 실행합니다.
pg.homoscedasticity(data=store_2025, dv='sales', group='store_type', method='levene')

**해석**
- 일반 분산분석도 t-검정처럼 집단별로 정규성과 등분산성을 가정합니다.
  - 등분산 가정이 필요 없는 Welch 분산분석을 기본으로 사용합니다.
- 소형과 중형은 정규성을 만족하지만, 대형은 `normal`이 False입니다.
  - 대형 점포끼리 면적 차이가 커서 매출이 큰 점포 몇 곳이 분포를 한쪽으로 늘어뜨리기 때문입니다.
- 분산분석은 집단별 표본이 수십 개이고 크기가 비슷하면 정규성에서 조금 벗어나도 결과가 크게 흔들리지 않습니다.
  - 순위로 비교하는 Kruskal-Wallis 검정은 평균이 아니라 '어느 집단의 값이 큰 쪽에 몰려 있는가'를 보는 다른 질문이므로, 아래에서 함께 확인합니다.
- `equal_var`가 False이므로 세 유형의 분산이 같다고 보기 어렵습니다.
  - 일반 분산분석을 먼저 실행한 후, 등분산 가정이 필요 없는 Welch 분산분석의 결과로 판단합니다.

**질문**: 일반 분산분석으로 세 유형의 평균 매출이 모두 같다는 가설을 검정합니다.

In [ ]:
# 점포 유형별 매출의 일반 분산분석을 실행합니다.
pg.anova(data=store_2025, dv='sales', between='store_type')

**해석**
- 검정 결과 표에서 `F`는 검정통계량, `p_unc`는 유의확률입니다.
  - 유의확률이 0.05보다 작으면 '세 유형의 평균이 모두 같다'는 귀무가설을 기각합니다.
- 분산분석의 대립가설은 **적어도 한 쌍의 평균이 다르다**는 것입니다.
  - 위 결과는 세 집단의 평균이 모두 같지는 않다는 것만 알려 주고, 어느 쌍이 다른지는 알려 주지 않습니다.
  - 사후분석을 통해 어느 쌍의 평균이 서로 다른지 확인할 수 있습니다.
- `np2`(부분 에타제곱)는 매출 변동 중 점포 유형이 설명하는 비율(효과 크기)입니다.

**질문**: 등분산을 가정하지 않는 Welch 분산분석으로 점포 유형별 평균 매출을 비교하면 결론이 같은가?

In [ ]:
# 점포 유형별 매출의 Welch 분산분석을 실행합니다.
pg.welch_anova(data=store_2025, dv='sales', between='store_type')

**해석**
- 앞의 피벗 테이블을 보면 대형 점포의 매출이 다른 유형보다 훨씬 넓게 흩어져 있습니다.
  - 등분산성 검정에서 분산이 다르다고 나온 것과 같은 결과입니다.
- 일반 분산분석과 Welch 분산분석 모두 유의확률이 0.05보다 작아 결론이 같습니다.
  - 세 집단 이상에서도 등분산을 가정하지 않는 Welch 분산분석을 기본으로 사용합니다.

**질문**: 순위로 비교하는 Kruskal-Wallis 검정에서도 점포 유형별 매출에 차이가 있는가?

In [ ]:
# 점포 유형별 매출의 Kruskal-Wallis 검정을 실행합니다.
pg.kruskal(data=store_2025, dv='sales', between='store_type')

**해석**
- Kruskal-Wallis 검정은 Mann-Whitney U 검정을 세 집단 이상으로 넓힌 방법입니다.
  - 값 대신 **순위**를 비교합니다.
- 검정 결과 표에서 `H`는 검정통계량, `p_unc`는 유의확률입니다.
  - 유의확률이 0.05보다 작으므로 순위로 봐도 점포 유형별 매출에 차이가 있습니다.
- 평균(분산분석)으로 봐도, 순위(Kruskal-Wallis 검정)로 봐도 점포 유형별 매출 차이는 분명합니다.

**질문**: Tukey's HSD 사후분석으로 어느 유형끼리 차이가 나는지 확인합니다.

In [ ]:
# 점포 유형별 매출의 Tukey's HSD 사후분석을 실행합니다.
pg.pairwise_tukey(data=store_2025, dv='sales', between='store_type')

**해석**
- 검정 결과 표에서 `diff`는 두 유형의 평균 차이(A − B), `p_tukey`는 다중비교를 보정한 유의확률입니다.
  - `p_tukey`가 0.05보다 작은 쌍이 유의한 차이입니다.
- Tukey's HSD는 가장 큰 평균 차이의 분포(스튜던트화 범위 분포)를 기준으로 판단합니다.
  - 여러 쌍을 동시에 비교해도 제1종 오류를 0.05로 유지합니다.
- 독립표본 t-검정을 세 번 따로 하는 것보다 사후분석을 한 번 실행하는 것이 더 안전합니다.
  - 검정을 여러 번 반복하면 한 번이라도 잘못 기각할 확률(제1종 오류)이 0.05보다 커지기 때문입니다.

**질문**: 등분산을 가정하지 않는 Games-Howell 검정으로 점포 유형별 사후분석을 하면 결론이 같은가?

In [ ]:
# 점포 유형별 매출의 Games-Howell 사후분석을 실행합니다.
pg.pairwise_gameshowell(data=store_2025, dv='sales', between='store_type')

**해석**
- Welch 분산분석 다음에는 등분산을 가정하지 않는 Games-Howell 검정으로 사후분석을 합니다.
  - `diff`는 두 유형의 평균 차이(A − B)이고, `pval`은 다중비교를 보정한 유의확률입니다.
  - `pval`이 모두 0.05보다 작아, Tukey's HSD와 결론이 같습니다.

**질문**: Kruskal-Wallis 검정 다음에는 Dunn 검정으로 어느 유형끼리 차이가 나는지 확인합니다.

pingouin에는 Dunn 검정이 없으므로 함수를 직접 정의합니다.<br>
Dunn 검정은 전체 데이터의 순위를 매긴 뒤, 두 집단의 평균 순위 차이를 표준오차로 나눈 z값으로 비교합니다.<br>
여러 쌍을 비교하므로 Holm 방법으로 유의확률을 보정합니다.

In [ ]:
# Dunn 검정을 실행하는 dunn_test()를 정의합니다.
def dunn_test(data, dv, between):

    # dv 전체의 순위를 매깁니다. 같은 값에는 평균 순위를 부여합니다.
    ranks = data[dv].rank(method='average')
    
    # 집단별 표본 크기와 평균 순위를 계산합니다.
    summary = ranks.groupby(by=data[between], observed=True).agg(func=['count', 'mean'])
    
    # 전체 순위의 분산을 계산합니다. 같은 값(동순위)이 있으면 그만큼 분산을 줄여 보정합니다.
    n_total = len(ranks)
    ties = ranks.value_counts()
    rank_var = n_total * (n_total + 1) / 12 - (ties ** 3 - ties).sum() / (12 * (n_total - 1))
    
    # 두 집단씩 평균 순위의 차이를 표준오차로 나눈 z값과 양측 유의확률을 계산합니다.
    rows = []
    groups = summary.index.tolist()
    for i, a in enumerate(groups):
        for b in groups[i + 1:]:
            se = (rank_var * (1 / summary.loc[a, 'count'] + 1 / summary.loc[b, 'count'])) ** 0.5
            z = (summary.loc[a, 'mean'] - summary.loc[b, 'mean']) / se
            rows.append({'A': a, 'B': b, 
                         'mean_rank_A': summary.loc[a, 'mean'], 
                         'mean_rank_B': summary.loc[b, 'mean'], 
                         'z': z, 'p_unc': 2 * stats.norm.sf(x=abs(z))})
    result = pd.DataFrame(data=rows)
    
    # Holm 방법으로 유의확률을 보정합니다.
    result['p_holm'] = pg.multicomp(pvals=result['p_unc'], method='holm')[1]
    return result

In [ ]:
# 점포 유형별 매출의 Dunn 사후분석을 실행합니다.
dunn_test(data=store_2025, dv='sales', between='store_type')

**해석**
- 검정 결과 표의 `z`는 평균 순위 차이를 표준오차로 나눈 값이고, `p_holm`은 Holm 방법으로 보정한 유의확률입니다.
  - 모든 쌍에서 `p_holm`이 0.05보다 작으므로 Tukey's HSD 및 Games-Howell 검정과 결론이 같습니다.
  - 평균 순위가 소형, 중형, 대형 순으로 높아, 점포 규모가 클수록 매출 순위도 높다는 것을 의미합니다.
- Holm 방법은 유의확률이 작은 쌍부터 차례로 0.05를 남은 비교 횟수로 나눈 값과 비교합니다.
  - 모든 쌍을 0.05를 비교 횟수로 나눈 값과 비교하는 Bonferroni 방법보다 덜 엄격합니다.

**질문**: 같은 방법으로 지역별 평균 매출을 비교하면 어떤가?

In [ ]:
# 지역별 매출의 도수, 평균 및 표준편차를 확인합니다.
pd.pivot_table(data=store_2025, values='sales', index='region', aggfunc=['count', 'mean', 'std']) \
    .round(decimals=1)

In [ ]:
# 지역별 매출 분포를 상자 그림으로 시각화합니다.
plt.figure(figsize=(6, 4), dpi=100)
sns.boxplot(data=store_2025, x='region', y='sales', boxprops={'facecolor': 'silver'}, medianprops={'color': 'red'})
plt.title(label='지역별 월평균 매출 (2025년)')
plt.xlabel(xlabel='지역')
plt.ylabel(ylabel='월평균 매출 (백만 원)')
plt.show()

**해석**
- 수도권과 영남권은 중위수가 `3,600~3,700백만 원`으로 비슷하고, 충청·강원권과 호남권은 `2,500~2,600백만 원`으로 낮습니다.
  - 상자 그림에서 상자 가운데 빨간 선은 중위수입니다.
- 충청·강원권은 상자가 좁아 대부분 점포의 매출이 비슷하지만, 위쪽의 이상값(극단값) 세 곳이 모두 대형 점포입니다.
  - 충청·강원권의 평균이 중위수보다 크게 올라갑니다.
- 호남권은 상자가 가장 길고 중위수가 상자 아래쪽에 있어, 매출이 큰 점포 쪽으로 넓게 퍼져 있습니다.
  - 피벗 테이블에서 호남권의 표준편차가 가장 큰 이유입니다.
- 네 지역의 상자가 많이 겹치므로 상자 그림만으로는 지역 간 차이가 뚜렷하다고 보기 어렵습니다.
  - 지역 간 차이가 우연으로 설명되는 수준인지 검정으로 확인합니다.

**질문**: 분산분석 전에 지역별 매출의 정규성과 등분산성을 확인합니다.

In [ ]:
# 지역별 매출의 정규성 검정을 실행합니다.
pg.normality(data=store_2025, dv='sales', group='region')

In [ ]:
# 지역별 매출의 등분산성 검정을 실행합니다.
pg.homoscedasticity(data=store_2025, dv='sales', group='region', method='levene')

**해석**
- 영남권을 뺀 세 지역은 `normal`이 False입니다.
  - 한 지역 안에 소형·중형·대형 점포가 섞여 있어서 매출 분포가 한쪽으로 치우치거나 봉우리가 여러 개 생기기 때문입니다.
- Levene 검정은 이상값에 덜 민감한 중위수 기준으로 퍼짐을 비교합니다.
  - `equal_var`가 True이므로 지역별 분산이 다르다고 볼 근거는 없습니다.
  - 충청·강원권과 호남권은 실제로 퍼짐이 크지만, 중위수 기준으로는 차이가 없다고 판단한 것입니다.
- 정규성을 만족하지 않는 지역이 많으므로 일반 분산분석과 Welch 분산분석의 결론이 같은지 함께 확인합니다.

In [ ]:
# 지역별 매출의 일반 분산분석을 실행합니다.
pg.anova(data=store_2025, dv='sales', between='region')

**해석**
- 유의확률이 0.05보다 크므로 '지역별 평균 매출 차이가 있다'고 말할 수 없습니다.
  - 지역 안에서 점포 크기 차이가 워낙 커서(표준편차 열), 지역 간 차이가 그 변동에 묻혀 버렸기 때문입니다.

**질문**: 등분산을 가정하지 않는 Welch 분산분석으로 지역별 평균 매출을 비교하면 결론이 같은가?

In [ ]:
# 지역별 매출의 Welch 분산분석을 실행합니다.
pg.welch_anova(data=store_2025, dv='sales', between='region')

**해석**
- 일반 분산분석과 Welch 분산분석 모두 유의확률이 0.05보다 커서 결론이 같습니다.
  - 지역별 평균 매출이 다르다고 볼 근거가 부족합니다.

**질문**: 순위로 비교하는 Kruskal-Wallis 검정으로 지역별 매출을 비교하면 어떤가?

In [ ]:
# 지역별 매출의 Kruskal-Wallis 검정을 실행합니다.
pg.kruskal(data=store_2025, dv='sales', between='region')

In [ ]:
# 지역별 점포 유형 구성을 확인합니다.
pd.crosstab(index=store_2025['region'], columns=store_2025['store_type'])

**해석**
- 분산분석은 집단의 **평균**을 비교합니다.
  - 호남권과 충청·강원권은 매출이 아주 큰 대형 점포 몇 곳 때문에 표준편차가 커서, 평균 차이가 그 변동에 묻혔습니다.
- Kruskal-Wallis 검정은 **순위**를 비교하며, 유의확률이 0.05보다 작습니다.
  - 정리하면 '지역별 평균 차이의 근거는 부족하지만, 순위(분포) 차이의 근거는 있다'입니다.
  - 두 검정은 서로 다른 질문에 답하므로, 같은 질문에 대한 판정이 엇갈린 것이 아닙니다.
- 다만 교차표를 보면 충청·강원권은 소형 점포가 많고 대형 점포가 적습니다.
  - 그래서 지역 차이처럼 보이는 것의 일부는 지역마다 점포 규모 구성이 다르기 때문일 수 있습니다.
- 평균과 순위의 결과가 다르면 한쪽 결과만 보고 결론 내리지 않고, 관심 지표가 무엇인지와 데이터 구성을 함께 확인합니다.
  - 점포 크기를 맞춰 놓고 지역을 비교하는 방법은 선형 회귀 실습에서 다룹니다.

**질문**: 지역 Tukey's HSD 사후분석에서도 유의한 쌍이 없는가?

In [ ]:
# 지역별 매출의 Tukey's HSD 사후분석을 실행합니다.
pg.pairwise_tukey(data=store_2025, dv='sales', between='region')

**해석**
- 모든 쌍에서 `p_tukey`가 0.05보다 큽니다.
  - 어느 두 지역을 골라도 평균 매출 차이가 있다고 말할 수 없습니다.
- 그렇다면 지역은 매출과 정말 관계가 없을까요?
  - **선형 회귀 실습에서 매장 면적을 함께 고려(통제)하면 결과가 달라지는 것을 다시 확인합니다.**

**질문**: 등분산을 가정하지 않는 Games-Howell 검정으로 지역별 사후분석을 하면 결론이 같은가?

In [ ]:
# 지역별 매출의 Games-Howell 사후분석을 실행합니다.
pg.pairwise_gameshowell(data=store_2025, dv='sales', between='region')

**해석**
- 모든 쌍에서 `pval`이 0.05보다 커서, Tukey's HSD와 결론이 같습니다.
  - 어느 두 지역도 평균 매출이 다르다고 볼 근거가 부족합니다.
- Welch 분산분석 다음에는 이처럼 등분산을 가정하지 않는 Games-Howell 검정으로 사후분석을 합니다.

**질문**: Kruskal-Wallis 검정에서는 지역별 차이가 유의했습니다. Dunn 검정으로 어느 지역끼리 차이가 나는지 확인합니다.

In [ ]:
# 지역별 매출의 Dunn 사후분석을 실행합니다.
dunn_test(data=store_2025, dv='sales', between='region').sort_values(by='p_unc', ignore_index=True)

**해석**
- 수도권과 충청·강원권 한 쌍만 `p_holm`이 0.05보다 작습니다.
  - 교차표에서 본 것처럼 소형 점포가 많고 대형 점포가 적기 때문에 충청·강원권은 평균 순위가 가장 낮습니다.
- Tukey's HSD와 Games-Howell 검정은 평균을 비교하지만, Dunn 검정은 순위를 비교하므로 대형 점포 몇 곳의 극단값에 덜 흔들립니다.
  - Kruskal-Wallis 검정이 유의했던 것은 충청·강원권 점포가 매출 순위의 아래쪽에 몰려 있기 때문입니다.
  - 분산분석이 유의하지 않았던 것은 대형 점포 몇 곳이 충청·강원권의 평균과 표준편차를 함께 끌어올려, 평균 차이가 작아지고 지역 안 변동에 묻혔기 때문입니다.
- 수도권과 호남권, 영남권과 충청·강원권은 `p_unc`(보정 전 유의확률)이 0.05보다 작지만, Holm 방법으로 보정하면 유의하지 않습니다.
  - 여러 쌍을 비교할 때 유의확률을 보정해야 하는 이유입니다.

## 4. 두 범주형 변수의 관련성: 교차분석(카이제곱 검정)

두 범주형 변수 사이에 관계가 있는지 확인합니다.<br>
신규 고객의 `repurchase_m7`(7차월 재구매 여부)를 사용합니다.

**질문**: `main_channel`(주 이용 채널)에 따라 재구매율이 다른가?

In [ ]:
# 채널별 재구매율(행 기준 상대도수)을 확인합니다.
pd.crosstab(index=repurchase['main_channel'], columns=repurchase['repurchase_m7'], normalize='index', margins=True) \
    .round(decimals=3)

In [ ]:
# 교차분석(카이제곱 검정)을 실행합니다.
expected, observed, result = pg.chi2_independence(data=repurchase, x='main_channel', y='repurchase_m7')

# 카이제곱 검정 결과의 첫 행을 확인합니다.
# [참고] 여러 방법의 결과가 행으로 나오며, test가 'pearson'인 첫 행이 Pearson 카이제곱 검정입니다.
result.loc[[0], :]

**해석**
- 채널별 재구매율은 슈퍼 32.7%, 마트 35.0%, 온라인 36.5%로 가장 큰 차이가 약 3.8%p입니다.
  - 고객 수가 많아 작은 차이도 유의하게 나오므로 유의확률과 함께 차이의 크기를 봅니다.
- Pearson 카이제곱 검정 결과 `pval`이 0.05보다 작으므로 귀무가설을 기각합니다.
  - 채널과 재구매 여부가 서로 관련이 있다고 봅니다.
- `cramer`(Cramer's V)는 두 범주형 변수 간 관련성의 크기입니다.
  - 0.028로 0에 가까우므로 관련성이 매우 약합니다.

**질문**: `age_group`(연령대)에 따라 재구매율이 다른가?

In [ ]:
# 연령대별 재구매율(행 기준 상대도수)을 확인합니다.
pd.crosstab(index=repurchase['age_group'], columns=repurchase['repurchase_m7'], normalize='index') \
    .round(decimals=3)

In [ ]:
# 교차분석(카이제곱 검정)을 실행합니다.
expected, observed, result = pg.chi2_independence(data=repurchase, x='age_group', y='repurchase_m7')

# 카이제곱 검정 결과의 첫 행을 확인합니다.
result.loc[[0], :]

**해석**
- 유의확률이 0.05보다 크므로 귀무가설을 기각할 수 없습니다.
  - 연령대별 재구매율 차이는 우연으로 설명할 수 있는 수준입니다.
- '연령대별로 재구매 마케팅을 다르게 해야 한다'는 근거는 현재 데이터에서 찾기 어렵습니다.
  - 로지스틱 회귀 실습에서 다른 변수와 함께 다시 확인합니다.

## 정리

- 등분산 가정이 필요한지에 따라 Student t-검정과 Welch t-검정을 구분합니다.
  - 이번 실습에서는 Welch t-검정을 기본으로 판단했습니다.
  - 소형·중형 점포의 매출은 정규성과 등분산성을 모두 만족해 Student t-검정도 같은 결론이었고, 순위로 비교한 Mann-Whitney U 검정에서도 중형 점포의 매출이 큰 경향이 뚜렷했습니다.
  - 중형 점포의 평균 매출이 소형 점포보다 유의하게 컸고, 효과 크기도 매우 컸습니다.
- 대형까지 더한 분산분석으로 점포 유형별 매출 차이를 확인했고, 어느 쌍이 다른지는 사후분석으로 확인했습니다.
  - 이번 실습에서는 Welch 분산분석을 기본으로 판단했습니다.
- 지역별 매출은 평균 차이(분산분석)의 근거는 부족했지만, 순위 차이(Kruskal-Wallis 검정)의 근거는 있었습니다.
  - Dunn 검정으로 보면 수도권과 충청·강원권의 차이입니다.
  - 지역마다 점포 규모 구성이 달라서 생긴 차이일 수 있으므로 점포 크기를 맞춘 비교가 필요합니다.
- 카이제곱 검정으로 주 이용 채널과 재구매 여부의 관련성을 확인했지만, Cramer's V가 작아 관련성은 약합니다.
  - 연령대와 재구매 여부의 관련성은 확인되지 않았습니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.

- 점포 매출은 소형·중형·대형 규모에 따라 확연히 다르고, 그 차이는 우연으로 설명할 수 있는 수준이 아닙니다.
  - 점포를 평가하거나 비교할 때는 같은 규모끼리 비교해야 합니다.
- 지역별 평균 매출은 차이가 뚜렷하지 않지만, 점포를 매출 순으로 줄 세우면 충청·강원권 점포가 아래쪽에 몰려 있습니다.
  - 지역마다 소형·중형·대형 점포의 구성이 다르기 때문일 수 있으므로, 지역 비교는 점포 크기를 맞춰 놓고 해야 합니다.
- 주로 이용하는 채널에 따라 재구매율이 다르지만, 그 차이는 몇 %p 수준으로 크지 않습니다.
  - 연령대에 따른 재구매율 차이는 확인되지 않았습니다.

## End of Document